In [ ]:
# ============================================================
# STEP 0 — IMPORT LIBRARIES & UPLOAD DATA
# ============================================================
import json
import os
from collections import defaultdict
from sklearn.model_selection import train_test_split
from google.colab import files

print("Upload marathi_wsd_dataset_filtered.json")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Load dataset
with open(file_name, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} rows")



Upload marathi_wsd_dataset_filtered.json


Saving marathi_wsd_dataset_filtered (1).json to marathi_wsd_dataset_filtered (1) (1).json
Loaded 50242 rows


In [ ]:
# ============================================================
# STEP 1 — GROUP DATA BY (SENTENCE, WORD)
# ============================================================
# Why?
# Each sentence has multiple rows (different senses)
# We keep them together to avoid data leakage

sentence_groups = defaultdict(list)

for row in data:
    key = (row["sentence"], row["lemma"])  # group by sentence + word
    sentence_groups[key].append(row)

# Convert to list for splitting
group_keys = list(sentence_groups.keys())


In [ ]:

# ============================================================
# STEP 2 — FIND TRUE SENSE (LABEL = 1)
# ============================================================
# Why?
# Each sentence has exactly ONE correct sense (label = 1)
# We use it for stratified splitting

true_senses = []

for key in group_keys:
    rows = sentence_groups[key]

    # Find the row where label = 1
    label1_rows = [r for r in rows if r["label"] == 1]

    # Safety check
    if not label1_rows:
        raise ValueError(
            f"No label=1 found for sentence: {key[0][:50]}"
        )

    # Extract correct sense
    true_sense_id = label1_rows[0]["sense_id"]

    # Combine word + sense for stratification
    true_senses.append(f"{key[1]}_{true_sense_id}")



In [ ]:
# ============================================================
# STEP 3 — SPLIT GROUPS INTO TRAIN (70%) + TEMP (30%)
# ============================================================

from collections import Counter

# Count how many times each sense appears
sense_counts = Counter(true_senses)

# Separate groups into:
# - rare: senses with only 1 sentence → go directly to train
# - normal: senses with 2+ sentences → stratified split

normal_keys, normal_senses = [], []
rare_keys = []

for key, sense in zip(group_keys, true_senses):
    if sense_counts[sense] < 2:
        rare_keys.append(key)        # too few to split → force to train
    else:
        normal_keys.append(key)
        normal_senses.append(sense)

print(f"Normal groups (can split): {len(normal_keys)}")
print(f"Rare groups (→ train only): {len(rare_keys)}")

# Print which senses are rare so you can check
rare_senses = [s for s in sense_counts if sense_counts[s] < 2]
print(f"\nRare senses found:")
for s in rare_senses:
    print(f"  ⚠ {s}: only {sense_counts[s]} sentence")

# Stratified split on normal groups only
train_keys, temp_keys, train_senses, temp_senses = train_test_split(
    normal_keys,
    normal_senses,
    test_size=0.30,
    random_state=42,
    stratify=normal_senses
)

# Add rare keys to train
train_keys = list(train_keys) + rare_keys
print(f"\nAfter adding rare → Train groups: {len(train_keys)}")


Normal groups (can split): 14608
Rare groups (→ train only): 2

Rare senses found:
  ⚠ डोळा_105: only 1 sentence
  ⚠ वाटणे_403: only 1 sentence

After adding rare → Train groups: 10227


In [ ]:
# ============================================================
# STEP 4 — SPLIT TEMP INTO VALIDATION (15%) + TEST (15%)
# ============================================================

val_keys, test_keys, _, _ = train_test_split(
    temp_keys,
    temp_senses,
    test_size=0.50,
    random_state=42,
    stratify=temp_senses
)

In [ ]:
# ============================================================
# STEP 5 — FLATTEN GROUPS BACK TO ROW FORMAT
# ============================================================
# Each group (sentence) contains multiple rows
# We convert grouped data back into normal dataset format

train_data = [row for key in train_keys for row in sentence_groups[key]]
val_data   = [row for key in val_keys   for row in sentence_groups[key]]
test_data  = [row for key in test_keys  for row in sentence_groups[key]]



In [ ]:
# ============================================================
# STEP 6 — SAVE SPLITS INTO FILES
# ============================================================

os.makedirs("splits", exist_ok=True)

for name, dataset in [("train", train_data),
                      ("val", val_data),
                      ("test", test_data)]:

    path = f"splits/{name}.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    print(f"✅ {name.upper():5s}: {len(dataset):6,} rows → {path}")
    files.download(path)


✅ TRAIN: 35,171 rows → splits/train.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ VAL  :  7,529 rows → splits/val.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ TEST :  7,542 rows → splits/test.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# STEP 7 — VERIFICATION (CHECK 70/15/15 SPLIT PER SENSE)
# ============================================================

print("\n── Per-sense label=1 verification ─────────────────────")

print(f"{'Word':<12} {'Sense':>7}  {'Train':>6}  {'Val':>5}  {'Test':>5}  {'Total':>6}  {'Train%':>7}")
print("─" * 60)


# Count only label=1 (correct senses)
def count_label1(rows):
    c = defaultdict(int)
    for r in rows:
        if r["label"] == 1:
            c[(r["lemma"], r["sense_id"])] += 1
    return c


# Get counts
train_c = count_label1(train_data)
val_c   = count_label1(val_data)
test_c  = count_label1(test_data)


# Combine all keys
all_keys = sorted(set(train_c) | set(val_c) | set(test_c))


# Print report
for lemma, sense_id in all_keys:
    t = train_c.get((lemma, sense_id), 0)
    v = val_c.get((lemma, sense_id), 0)
    te = test_c.get((lemma, sense_id), 0)

    total = t + v + te
    pct = f"{100*t/total:.1f}%" if total else "N/A"

    print(f"{lemma:<12} {sense_id:>7}  {t:>6}  {v:>5}  {te:>5}  {total:>6}  {pct:>7}")




── Per-sense label=1 verification ─────────────────────
Word           Sense   Train    Val   Test   Total   Train%
────────────────────────────────────────────────────────────
अंक             2001      69     15     14      98    70.4%
अंक             2002      67     15     15      97    69.1%
अंक             2003      68     15     15      98    69.4%
अर्थ             801      92     20     20     132    69.7%
अर्थ             802     206     44     44     294    70.1%
अर्थ             803      78     20     17     115    67.8%
गुण             2101      80     18     17     115    69.6%
गुण             2102      85     14     16     115    73.9%
गुण             2103      82     17     16     115    71.3%
डोळा             101      64     14     14      92    69.6%
डोळा             102      68     15     15      98    69.4%
डोळा             103      74     16     16     106    69.8%
डोळा             104      73     17     16     106    68.9%
डोळा             105      64     14     14

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print(f"\nGrand totals:")
print(f"Train : {len(train_data):,}")
print(f"Val   : {len(val_data):,}")
print(f"Test  : {len(test_data):,}")


Grand totals:
Train : 35,171
Val   : 7,529
Test  : 7,542


In [ ]:
# ============================================================
# STEP 7 — VERIFICATION (BOTH LABEL=1 AND LABEL=0)
# ============================================================

def count_by_label(rows, label):
    c = defaultdict(int)
    for r in rows:
        if r["label"] == label:
            c[(r["lemma"], r["sense_id"])] += 1
    return c

# Count label=1 (correct sense)
train_1 = count_by_label(train_data, 1)
val_1   = count_by_label(val_data,   1)
test_1  = count_by_label(test_data,  1)

# Count label=0 (wrong senses)
train_0 = count_by_label(train_data, 0)
val_0   = count_by_label(val_data,   0)
test_0  = count_by_label(test_data,  0)

all_keys = sorted(set(train_1) | set(val_1) | set(test_1))

# ── Label=1 table ──────────────────────────────────────────
print("\n── Label=1 (Correct Sense) Verification ───────────────")
print(f"{'Word':<12} {'Sense':>7}  {'Train':>6}  {'Val':>5}  {'Test':>5}  {'Total':>6}  {'Train%':>7}")
print("─" * 60)

for lemma, sense_id in all_keys:
    t  = train_1.get((lemma, sense_id), 0)
    v  = val_1.get((lemma, sense_id), 0)
    te = test_1.get((lemma, sense_id), 0)
    total = t + v + te
    pct = f"{100*t/total:.1f}%" if total else "N/A"
    print(f"{lemma:<12} {sense_id:>7}  {t:>6}  {v:>5}  {te:>5}  {total:>6}  {pct:>7}")

# ── Label=0 table ──────────────────────────────────────────
print("\n── Label=0 (Wrong Sense / Negative) Verification ──────")
print(f"{'Word':<12} {'Sense':>7}  {'Train':>6}  {'Val':>5}  {'Test':>5}  {'Total':>6}  {'Train%':>7}")
print("─" * 60)

for lemma, sense_id in all_keys:
    t  = train_0.get((lemma, sense_id), 0)
    v  = val_0.get((lemma, sense_id), 0)
    te = test_0.get((lemma, sense_id), 0)
    total = t + v + te
    pct = f"{100*t/total:.1f}%" if total else "N/A"
    print(f"{lemma:<12} {sense_id:>7}  {t:>6}  {v:>5}  {te:>5}  {total:>6}  {pct:>7}")

# ── Grand totals ───────────────────────────────────────────
print(f"\n── Grand Totals ────────────────────────────────────────")
print(f"{'Split':<8} {'Label=1':>8}  {'Label=0':>8}  {'Total':>8}")
print("─" * 38)

for name, pos, neg in [
    ("Train", sum(train_1.values()), sum(train_0.values())),
    ("Val",   sum(val_1.values()),   sum(val_0.values())),
    ("Test",  sum(test_1.values()),  sum(test_0.values())),
]:
    print(f"{name:<8} {pos:>8,}  {neg:>8,}  {pos+neg:>8,}")


── Label=1 (Correct Sense) Verification ───────────────
Word           Sense   Train    Val   Test   Total   Train%
────────────────────────────────────────────────────────────
अंक             2001      69     15     14      98    70.4%
अंक             2002      67     15     15      97    69.1%
अंक             2003      68     15     15      98    69.4%
अर्थ             801      92     20     20     132    69.7%
अर्थ             802     206     44     44     294    70.1%
अर्थ             803      78     20     17     115    67.8%
गुण             2101      80     18     17     115    69.6%
गुण             2102      85     14     16     115    73.9%
गुण             2103      82     17     16     115    71.3%
डोळा             101      64     14     14      92    69.6%
डोळा             102      68     15     15      98    69.4%
डोळा             103      74     16     16     106    69.8%
डोळा             104      73     17     16     106    68.9%
डोळा             105      64     14     14